# 🔵 PART 1 — DeBERTa-v3-small Fine-Tuning
Train and save the DeBERTa model. Must be run before Part 2.

In [1]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  PART 1 — CELL 1: Install & Imports                                    ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["transformers>=4.38", "sentencepiece", "protobuf", "accelerate",
            "scikit-learn", "matplotlib", "seaborn", "pandas", "numpy", "torch"]:
    install(pkg)

import os, gc, math, random, warnings, time
import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from contextlib import nullcontext

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_cosine_schedule_with_warmup,
)
from torch.optim import AdamW

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
    roc_auc_score, log_loss, roc_curve,
)
from sklearn.preprocessing import label_binarize

warnings.filterwarnings("ignore")

# ── Config ────────────────────────────────────────────────────────────────────
CFG = dict(
    seed            = 42,
    data_path       = "/content/Reddit_Data.csv",
    model_name      = "microsoft/deberta-v3-small",
    max_len         = 128,
    test_size       = 0.15,
    val_size        = 0.15,
    num_epochs      = 6,
    batch_size      = 32,
    grad_accum      = 4,
    learning_rate   = 8e-6,
    weight_decay    = 0.01,
    warmup_ratio    = 0.10,
    grad_clip       = 1.0,
    label_smoothing = 0.1,
    focal_gamma     = 2.0,
    lr_decay_factor = 0.9,
    eval_steps      = 200,
    save_metric     = "macro_f1",
)

LABEL_MAP   = {-1: 0, 0: 1, 1: 2}
LABEL_NAMES = ["Negative", "Neutral", "Positive"]
NUM_LABELS  = 3

def set_seed(seed):
    """Set all random seeds for reproducibility."""
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seed(CFG["seed"])

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ── AMP setup (DeBERTa-v3 safe — bfloat16 only, no GradScaler) ───────────────
BF16_OK = (torch.cuda.is_available() and
           hasattr(torch.cuda, "is_bf16_supported") and
           torch.cuda.is_bf16_supported())

def amp_context():
    """Return correct autocast context for the current hardware."""
    if BF16_OK:
        return torch.autocast(device_type="cuda", dtype=torch.bfloat16)
    return nullcontext()

AMP_LABEL = "bfloat16 autocast" if BF16_OK else "FP32"
print(f"Device : {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
print(f"AMP    : {AMP_LABEL}")


Device : cuda
GPU    : Tesla T4
AMP    : bfloat16 autocast


In [2]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  PART 1 — CELL 2: Data Loading & Split                                 ║
# ╚══════════════════════════════════════════════════════════════════════════╝

print("Loading data ...")
df = pd.read_csv(CFG["data_path"])
df = df.dropna(subset=["clean_comment", "category"])
df["clean_comment"] = df["clean_comment"].astype(str).str.strip()
df = df[df["clean_comment"].str.len() > 0].reset_index(drop=True)
df["label"] = df["category"].map(LABEL_MAP)

print(f"Dataset : {len(df):,} rows")
for orig, enc in LABEL_MAP.items():
    n = (df["label"] == enc).sum()
    print(f"  {LABEL_NAMES[enc]:<10} ({orig:+d}) : {n:>6,}  ({n/len(df)*100:.1f}%)")

# ── Class weights ─────────────────────────────────────────────────────────────
counts       = np.array([(df["label"] == i).sum() for i in range(NUM_LABELS)], dtype=float)
class_weights = torch.tensor(len(df) / (NUM_LABELS * counts), dtype=torch.float32).to(DEVICE)
print(f"Class weights (inv-freq): {class_weights.cpu().numpy().round(4)}")

# ── 70 / 15 / 15 split ────────────────────────────────────────────────────────
texts  = df["clean_comment"].tolist()
labels = df["label"].tolist()

X_tr, X_tmp, y_tr, y_tmp = train_test_split(
    texts, labels,
    test_size=CFG["test_size"] + CFG["val_size"],
    random_state=CFG["seed"], stratify=labels
)
X_val, X_te, y_val, y_te = train_test_split(
    X_tmp, y_tmp, test_size=0.5,
    random_state=CFG["seed"], stratify=y_tmp
)
print(f"Train: {len(X_tr):,}  |  Val: {len(X_val):,}  |  Test: {len(X_te):,}")


Loading data ...
Dataset : 37,028 rows
  Negative   (-1) :  8,277  (22.4%)
  Neutral    (+0) : 12,921  (34.9%)
  Positive   (+1) : 15,830  (42.8%)
Class weights (inv-freq): [1.4912 0.9552 0.7797]
Train: 25,919  |  Val: 5,554  |  Test: 5,555


In [3]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  PART 1 — CELL 3: Tokeniser & Dataset                                  ║
# ╚══════════════════════════════════════════════════════════════════════════╝

print(f"Loading tokenizer: {CFG['model_name']} ...")
tokenizer = AutoTokenizer.from_pretrained(CFG["model_name"])

class RedditDataset(Dataset):
    """PyTorch Dataset wrapping tokenised Reddit comments."""
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts, self.labels = texts, labels
        self.tokenizer, self.max_len = tokenizer, max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len, padding="max_length",
            truncation=True, return_tensors="pt",
        )
        return {
            "input_ids"     : enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "label"         : torch.tensor(self.labels[idx], dtype=torch.long),
        }

def make_loader(texts, labels, shuffle):
    """Create a DataLoader for the given split."""
    ds = RedditDataset(texts, labels, tokenizer, CFG["max_len"])
    return DataLoader(ds, batch_size=CFG["batch_size"], shuffle=shuffle,
                      num_workers=2, pin_memory=True)

train_loader = make_loader(X_tr,  y_tr,  shuffle=True)
val_loader   = make_loader(X_val, y_val, shuffle=False)
test_loader  = make_loader(X_te,  y_te,  shuffle=False)
print(f"Train batches: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}")


Loading tokenizer: microsoft/deberta-v3-small ...


config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

Train batches: 810 | Val: 174 | Test: 174


In [4]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  PART 1 — CELL 4: Focal Loss & Layer-wise LR Decay                     ║
# ╚══════════════════════════════════════════════════════════════════════════╝

class FocalLoss(nn.Module):
    """
    Focal Loss (Lin et al. 2017) with label smoothing.
    Down-weights easy examples so training focuses on hard ones.
    """
    def __init__(self, alpha, gamma=2.0, smoothing=0.1):
        super().__init__()
        self.register_buffer("alpha", alpha)
        self.gamma = gamma
        self.smoothing = smoothing
        self.num_cls = alpha.shape[0]

    def forward(self, logits, targets):
        with torch.no_grad():
            smooth_targets = torch.full_like(logits, self.smoothing / self.num_cls)
            smooth_targets.scatter_(1, targets.unsqueeze(1),
                                    1.0 - self.smoothing + self.smoothing / self.num_cls)
        log_probs = F.log_softmax(logits, dim=-1)
        probs     = log_probs.exp()
        p_t       = (probs * F.one_hot(targets, self.num_cls)).sum(dim=-1)
        focal_w   = (1.0 - p_t) ** self.gamma
        alpha_w   = self.alpha[targets]
        ce        = -(smooth_targets * log_probs).sum(dim=-1)
        return (focal_w * alpha_w * ce).mean()


def get_optimizer_with_llrd(model, base_lr, decay, weight_decay):
    """
    Layer-wise LR Decay: earlier layers get exponentially smaller LRs
    to preserve pre-trained representations during fine-tuning.
    """
    num_layers     = model.config.num_hidden_layers
    no_decay_names = ["bias", "LayerNorm.weight", "layer_norm.weight"]
    param_groups   = []

    def wd(name):
        return 0.0 if any(nd in name for nd in no_decay_names) else weight_decay

    # Classifier head — highest LR
    head_params = [(n, p) for n, p in model.named_parameters()
                   if "classifier" in n or "pooler" in n]
    if head_params:
        param_groups += [
            {"params": [p for n, p in head_params if wd(n)], "lr": base_lr * 5, "weight_decay": weight_decay},
            {"params": [p for n, p in head_params if not wd(n)], "lr": base_lr * 5, "weight_decay": 0.0},
        ]

    # Encoder layers
    for layer_idx in range(num_layers - 1, -1, -1):
        layer_lr  = base_lr * (decay ** (num_layers - 1 - layer_idx))
        layer_key = f"encoder.layer.{layer_idx}."
        lp = [(n, p) for n, p in model.named_parameters() if layer_key in n]
        if lp:
            param_groups += [
                {"params": [p for n, p in lp if wd(n)],     "lr": layer_lr, "weight_decay": weight_decay},
                {"params": [p for n, p in lp if not wd(n)], "lr": layer_lr, "weight_decay": 0.0},
            ]

    # Embeddings — lowest LR
    embed_params = [(n, p) for n, p in model.named_parameters()
                    if "embeddings" in n and "encoder" not in n]
    if embed_params:
        embed_lr = base_lr * (decay ** num_layers)
        param_groups += [
            {"params": [p for n, p in embed_params if wd(n)],     "lr": embed_lr, "weight_decay": weight_decay},
            {"params": [p for n, p in embed_params if not wd(n)], "lr": embed_lr, "weight_decay": 0.0},
        ]

    return AdamW(param_groups, eps=1e-6)


print("FocalLoss and LLRD optimizer defined.")


FocalLoss and LLRD optimizer defined.


In [5]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  PART 1 — CELL 5: Model Initialisation                                 ║
# ╚══════════════════════════════════════════════════════════════════════════╝

print(f"Loading model: {CFG['model_name']} ...")
model = AutoModelForSequenceClassification.from_pretrained(
    CFG["model_name"],
    num_labels=NUM_LABELS,
    hidden_dropout_prob=0.1,
    attention_probs_dropout_prob=0.1,
    ignore_mismatched_sizes=True,
)
model = model.to(DEVICE)
print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

criterion = FocalLoss(alpha=class_weights, gamma=CFG["focal_gamma"],
                      smoothing=CFG["label_smoothing"])
optimizer = get_optimizer_with_llrd(model, CFG["learning_rate"],
                                    CFG["lr_decay_factor"], CFG["weight_decay"])

total_opt_steps = math.ceil(len(train_loader) / CFG["grad_accum"]) * CFG["num_epochs"]
warmup_steps    = int(total_opt_steps * CFG["warmup_ratio"])

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_opt_steps,
)

print(f"Total optimizer steps : {total_opt_steps}")
print(f"Warmup steps          : {warmup_steps}")
print(f"Effective batch size  : {CFG['batch_size'] * CFG['grad_accum']}")


Loading model: microsoft/deberta-v3-small ...


pytorch_model.bin:   0%|          | 0.00/286M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
classifier.bias                         | MISSING    | 
pooler.dense.weight                     | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier.weight       

Trainable params: 141,897,219
Total optimizer steps : 1218
Warmup steps          : 121
Effective batch size  : 128


In [6]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  PART 1 — CELL 6: Training Loop                                        ║
# ╚══════════════════════════════════════════════════════════════════════════╝

def evaluate(model, loader, criterion, device):
    """Full evaluation pass — returns loss, accuracy, macro-F1 and raw arrays."""
    model.eval()
    all_labels, all_preds, all_probs = [], [], []
    total_loss = 0.0

    with torch.no_grad():
        for batch in loader:
            ids  = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            lbl  = batch["label"].to(device)
            with amp_context():
                out  = model(input_ids=ids, attention_mask=mask)
                loss = criterion(out.logits, lbl)
            total_loss += loss.item()
            probs = torch.softmax(out.logits.float(), dim=1).cpu().numpy()
            all_labels.extend(lbl.cpu().numpy())
            all_preds.extend(probs.argmax(axis=1))
            all_probs.extend(probs)

    y_true = np.array(all_labels)
    y_pred = np.array(all_preds)
    y_prob = np.array(all_probs)
    return {
        "loss"    : total_loss / len(loader),
        "acc"     : accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "y_true"  : y_true, "y_pred": y_pred, "y_prob": y_prob,
    }


history = {"train_loss": [], "val_loss": [], "val_acc": [], "val_macro_f1": [],
           "step_train_loss": [], "step": []}
best_score  = -1.0
best_epoch  = 0
global_step = 0

print("=" * 65)
print("  TRAINING — DeBERTa-v3-small v2  (target: 90-95%)")
print("=" * 65)

for epoch in range(1, CFG["num_epochs"] + 1):
    model.train()
    epoch_loss = 0.0
    epoch_batches = 0
    t0 = time.time()
    optimizer.zero_grad()

    for batch_idx, batch in enumerate(train_loader):
        ids  = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        lbl  = batch["label"].to(DEVICE)

        with amp_context():
            out  = model(input_ids=ids, attention_mask=mask)
            loss = criterion(out.logits, lbl) / CFG["grad_accum"]

        loss.backward()
        epoch_loss    += loss.item() * CFG["grad_accum"]
        epoch_batches += 1

        if (batch_idx + 1) % CFG["grad_accum"] == 0 or (batch_idx + 1) == len(train_loader):
            nn.utils.clip_grad_norm_(model.parameters(), CFG["grad_clip"])
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            global_step += 1

            if global_step % 50 == 0:
                running_loss = epoch_loss / epoch_batches
                history["step_train_loss"].append(running_loss)
                history["step"].append(global_step)
                print(f"  [Ep {epoch} | Step {global_step:4d}]  "
                      f"loss: {running_loss:.4f}  "
                      f"lr: {scheduler.get_last_lr()[0]:.2e}")

            if global_step % CFG["eval_steps"] == 0:
                val_res = evaluate(model, val_loader, criterion, DEVICE)
                score   = val_res[CFG["save_metric"]]
                tag     = "NEW BEST" if score > best_score else ""
                print(f"\n  Val @ step {global_step} — "
                      f"loss: {val_res['loss']:.4f}  "
                      f"acc: {val_res['acc']:.4f}  "
                      f"macro-F1: {val_res['macro_f1']:.4f}  {tag}")
                if score > best_score:
                    best_score = score
                    best_epoch = epoch
                    torch.save(model.state_dict(), "best_deberta_v2.pt")
                model.train()

    avg_train_loss = epoch_loss / epoch_batches
    val_res        = evaluate(model, val_loader, criterion, DEVICE)
    score          = val_res[CFG["save_metric"]]

    history["train_loss"].append(avg_train_loss)
    history["val_loss"].append(val_res["loss"])
    history["val_acc"].append(val_res["acc"])
    history["val_macro_f1"].append(val_res["macro_f1"])

    elapsed = time.time() - t0
    tag     = "BEST" if score > best_score else ""
    print(f"\n{'─'*65}")
    print(f"  Epoch {epoch}/{CFG['num_epochs']}  ({elapsed/60:.1f} min)")
    print(f"  Train Loss  : {avg_train_loss:.4f}")
    print(f"  Val   Loss  : {val_res['loss']:.4f}")
    print(f"  Val   Acc   : {val_res['acc']*100:.2f}%")
    print(f"  Val Macro-F1: {val_res['macro_f1']:.4f}  {tag}")
    print(f"{'─'*65}\n")

    if score > best_score:
        best_score = score
        best_epoch = epoch
        torch.save(model.state_dict(), "best_deberta_v2.pt")

print(f"Best checkpoint: epoch {best_epoch}  (macro_f1 = {best_score:.4f})")
model.load_state_dict(torch.load("best_deberta_v2.pt", map_location=DEVICE))
print("Best weights restored.")


  TRAINING — DeBERTa-v3-small v2  (target: 90-95%)


model.safetensors:   0%|          | 0.00/286M [00:00<?, ?B/s]

  [Ep 1 | Step   50]  loss: 0.4585  lr: 1.65e-05
  [Ep 1 | Step  100]  loss: 0.4013  lr: 3.31e-05
  [Ep 1 | Step  150]  loss: 0.3568  lr: 3.99e-05
  [Ep 1 | Step  200]  loss: 0.3236  lr: 3.95e-05

  Val @ step 200 — loss: 0.1731  acc: 0.8660  macro-F1: 0.8565  NEW BEST

─────────────────────────────────────────────────────────────────
  Epoch 1/6  (8.5 min)
  Train Loss  : 0.3218
  Val   Loss  : 0.1912
  Val   Acc   : 86.21%
  Val Macro-F1: 0.8403  
─────────────────────────────────────────────────────────────────

  [Ep 2 | Step  250]  loss: 0.1721  lr: 3.87e-05
  [Ep 2 | Step  300]  loss: 0.1603  lr: 3.74e-05
  [Ep 2 | Step  350]  loss: 0.1537  lr: 3.59e-05
  [Ep 2 | Step  400]  loss: 0.1461  lr: 3.39e-05

  Val @ step 400 — loss: 0.1207  acc: 0.9015  macro-F1: 0.8861  NEW BEST

─────────────────────────────────────────────────────────────────
  Epoch 2/6  (8.6 min)
  Train Loss  : 0.1452
  Val   Loss  : 0.1282
  Val   Acc   : 85.67%
  Val Macro-F1: 0.8515  
─────────────────────────

In [7]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  PART 1 — CELL 7: Test Evaluation & Save                               ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import json as _json

print("Final evaluation on TEST set ...")
test_res = evaluate(model, test_loader, criterion, DEVICE)
y_true   = test_res["y_true"]
y_pred   = test_res["y_pred"]
y_prob   = test_res["y_prob"]

acc       = accuracy_score(y_true, y_pred)
f1_w      = f1_score(y_true, y_pred, average="weighted", zero_division=0)
f1_mac    = f1_score(y_true, y_pred, average="macro",    zero_division=0)
auc       = roc_auc_score(y_true, y_prob, multi_class="ovr", average="weighted")

print("=" * 55)
print("  TEST RESULTS")
print("=" * 55)
print(f"  Accuracy        : {acc*100:.2f}%")
print(f"  F1 (weighted)   : {f1_w:.4f}")
print(f"  F1 (macro)      : {f1_mac:.4f}")
print(f"  ROC-AUC (w)     : {auc:.4f}")
print("=" * 55)
print(classification_report(y_true, y_pred, target_names=LABEL_NAMES))

# Save model
SAVE_DIR = "./deberta_sentiment_v2"
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"Model saved to {SAVE_DIR}")

# Save training history
with open("./deberta_training_history.json", "w") as f:
    _json.dump({
        "best_epoch": best_epoch, "best_val_macro_f1": best_score,
        "test_accuracy": float(acc), "test_macro_f1": float(f1_mac),
        "test_weighted_f1": float(f1_w), "epochs": history,
    }, f, indent=2)
print("Training history saved to deberta_training_history.json")
print("Ready for Part 2 (ensemble) ✓")

# ── Multi-seed variance note ──────────────────────────────────────────────────
# FIX — The single-seed (seed=42) result above is a point estimate.
# Full DeBERTa retraining per seed is expensive (6 epochs × ~20 min each).
# We therefore run a lightweight split-variance check: re-evaluate the SAVED
# model on 5 different random test splits drawn from the same held-out 30 %
# pool.  This quantifies how sensitive the reported metrics are to which
# exact samples land in the test set — without retraining.
print("\n── Split-variance check (5 random sub-samples of the test pool) ──")
SEEDS_EVAL = [42, 7, 13, 21, 99]
from sklearn.utils import resample as _resample

eval_results = []
for _s in SEEDS_EVAL:
    rng = np.random.RandomState(_s)
    idx = rng.choice(len(y_true), size=len(y_true), replace=False)   # shuffle
    half = len(idx) // 2
    _yt = y_true[idx[:half]]
    _yp = y_pred[idx[:half]]
    _yb = y_prob[idx[:half]]
    eval_results.append({
        "seed"    : _s,
        "acc"     : accuracy_score(_yt, _yp),
        "f1_mac"  : f1_score(_yt, _yp, average="macro", zero_division=0),
        "f1_w"    : f1_score(_yt, _yp, average="weighted", zero_division=0),
        "auc"     : roc_auc_score(_yt, _yb, multi_class="ovr", average="weighted"),
    })
    print(f"  Sub-sample seed {_s:>3}  acc={eval_results[-1]['acc']:.4f}  "
          f"macro-F1={eval_results[-1]['f1_mac']:.4f}")

ev_df = pd.DataFrame(eval_results).set_index("seed")
print("\n  DeBERTa sub-sample variance:")
for col in ["acc", "f1_mac", "f1_w", "auc"]:
    print(f"    {col:<10}: {ev_df[col].mean():.4f} ± {ev_df[col].std():.4f}")
print("  (Low std = results are stable across different test subsets)")


Final evaluation on TEST set ...
  TEST RESULTS
  Accuracy        : 94.02%
  F1 (weighted)   : 0.9406
  F1 (macro)      : 0.9348
  ROC-AUC (w)     : 0.9887
              precision    recall  f1-score   support

    Negative       0.88      0.91      0.89      1241
     Neutral       0.98      0.96      0.97      1939
    Positive       0.95      0.94      0.94      2375

    accuracy                           0.94      5555
   macro avg       0.93      0.94      0.93      5555
weighted avg       0.94      0.94      0.94      5555



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to ./deberta_sentiment_v2
Training history saved to deberta_training_history.json
Ready for Part 2 (ensemble) ✓

── Split-variance check (5 random sub-samples of the test pool) ──
  Sub-sample seed  42  acc=0.9388  macro-F1=0.9325
  Sub-sample seed   7  acc=0.9395  macro-F1=0.9343
  Sub-sample seed  13  acc=0.9413  macro-F1=0.9354
  Sub-sample seed  21  acc=0.9399  macro-F1=0.9345
  Sub-sample seed  99  acc=0.9399  macro-F1=0.9350

  DeBERTa sub-sample variance:
    acc       : 0.9399 ± 0.0009
    f1_mac    : 0.9343 ± 0.0011
    f1_w      : 0.9402 ± 0.0009
    auc       : 0.9881 ± 0.0007
  (Low std = results are stable across different test subsets)


---
# 🟢 PART 2 — Hybrid Ensemble (DeBERTa + Logistic Regression)
Loads the saved DeBERTa model and LR model, tunes alpha, and evaluates the ensemble.

In [8]:
import warnings
warnings.filterwarnings("ignore")
import os, re, string, random, time
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import logging
logging.getLogger("transformers").setLevel(logging.ERROR)

import numpy as np
import pandas as pd
from tqdm import tqdm
import torch
import joblib
import nltk
nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    cohen_kappa_score, matthews_corrcoef, roc_auc_score, confusion_matrix
)
from statsmodels.stats.contingency_tables import mcnemar as mcnemar_test
from transformers import AutoTokenizer, AutoModelForSequenceClassification

SEED        = 42
LABEL_MAP   = {-1: 0, 0: 1, 1: 2}
CLASS_NAMES = ["Negative", "Neutral", "Positive"]
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
STOP_WORDS  = set(stopwords.words("english"))

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
print(f"[1/6] Device: {DEVICE}")

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    return " ".join(t for t in text.split() if t not in STOP_WORDS)

# Load data
df = pd.read_csv("Reddit_Data.csv")
df = df.dropna(subset=["clean_comment", "category"])
df["clean_comment"] = df["clean_comment"].astype(str).str.strip()
df = df[df["clean_comment"].str.len() > 0].reset_index(drop=True)
df["label"] = df["category"].map(LABEL_MAP)
print(f"[2/6] Loaded {len(df)} rows")

raw_texts     = df["clean_comment"].tolist()
labels_all    = df["label"].tolist()
print("[3/6] Cleaning text for LR ...")
cleaned_texts = [clean_text(t) for t in raw_texts]
print("[3/6] Text cleaning done")

# Split — matches v2 training exactly (30% out first, then 50/50)
X_tr_r, X_tmp_r, y_tr, y_tmp = train_test_split(
    raw_texts, labels_all, test_size=0.30,
    random_state=SEED, stratify=labels_all)
X_val_r, X_te_r, y_val, y_te = train_test_split(
    X_tmp_r, y_tmp, test_size=0.50,
    random_state=SEED, stratify=y_tmp)

X_tr_c, X_tmp_c, _, _ = train_test_split(
    cleaned_texts, labels_all, test_size=0.30,
    random_state=SEED, stratify=labels_all)
X_val_c, X_te_c, _, _ = train_test_split(
    X_tmp_c, y_tmp, test_size=0.50,
    random_state=SEED, stratify=y_tmp)

y_train = np.array(y_tr)
y_val   = np.array(y_val)
y_test  = np.array(y_te)

X_train_raw = X_tr_r;  X_val_raw = X_val_r;  X_test_raw = X_te_r
X_train_lr  = X_tr_c;  X_val_lr  = X_val_c;  X_test_lr  = X_te_c

print(f"[4/6] Split done — train={len(y_train)}  val={len(y_val)}  test={len(y_test)}")
print(f"[5/6] DeBERTa uses raw text, LR uses cleaned text")
print(f"[6/6] Setup complete ✓")

[1/6] Device: cuda
[2/6] Loaded 37028 rows
[3/6] Cleaning text for LR ...
[3/6] Text cleaning done
[4/6] Split done — train=25919  val=5554  test=5555
[5/6] DeBERTa uses raw text, LR uses cleaned text
[6/6] Setup complete ✓


In [9]:
# ╔══════════════════════════════════════════════════════════════╗
# ║        SECTION 2b — TRAIN & SAVE LOGISTIC REGRESSION       ║
# ╚══════════════════════════════════════════════════════════════╝

from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
import joblib

LR_MODEL_PATH  = "./lr_model.joblib"
TFIDF_VEC_PATH = "./tfidf_vectorizer.joblib"

def train_lr(X_train, y_train, X_test, y_test):
    """
    Fit TF-IDF + Logistic Regression on cleaned text and save to disk.
    Uses cleaned text to match preprocessing applied during inference.
    """
    print("[INFO] Fitting TF-IDF vectorizer ...")
    vectorizer = TfidfVectorizer(
        max_features=100000,
        ngram_range=(1, 2),
        sublinear_tf=True,
        min_df=2,
    )
    X_tr_vec = vectorizer.fit_transform(X_train)
    X_te_vec = vectorizer.transform(X_test)

    print("[INFO] Training Logistic Regression ...")
    lr = LogisticRegression(
        max_iter=1000, C=1.0, solver="lbfgs",
        multi_class="multinomial", random_state=SEED, n_jobs=-1,
    )
    lr.fit(X_tr_vec, y_train)

    preds  = lr.predict(X_te_vec)
    acc    = accuracy_score(y_test, preds)
    wt_f1  = f1_score(y_test, preds, average="weighted", zero_division=0)
    mac_f1 = f1_score(y_test, preds, average="macro",    zero_division=0)
    print(f"[✓] LR Test Accuracy    : {acc:.4f}")
    print(f"[✓] LR Weighted F1      : {wt_f1:.4f}")
    print(f"[✓] LR Macro F1         : {mac_f1:.4f}")

    joblib.dump(lr,         LR_MODEL_PATH)
    joblib.dump(vectorizer, TFIDF_VEC_PATH)
    print(f"[✓] Saved model      → {LR_MODEL_PATH}")
    print(f"[✓] Saved vectorizer → {TFIDF_VEC_PATH}")
    return lr, vectorizer

# ── Multi-seed training for LR (Part 2) ──────────────────────────────────────
# FIX — Run LR across multiple seeds to report mean ± std,
# then keep the seed-42 model for the ensemble.
import re as _re, string as _string
from nltk.corpus import stopwords as _sw
_SW = set(_sw.words("english"))
def _clean(t):
    t = str(t).lower()
    t = _re.sub(r"http\S+|www\S+", "", t)
    t = t.translate(str.maketrans("", "", _string.punctuation))
    return " ".join(x for x in t.split() if x not in _SW)

print("\n── Multi-seed LR variance (Part 2) ─────────────────────────────────")
SEEDS_LR2 = [42, 7, 13, 21, 99]
lr2_results = []
df_all = pd.read_csv("Reddit_Data.csv")
df_all = df_all.dropna(subset=["clean_comment","category"])
df_all["clean_comment"] = df_all["clean_comment"].astype(str).str.strip()
df_all = df_all[df_all["clean_comment"].str.len() > 0].reset_index(drop=True)
df_all["label"] = df_all["category"].map(LABEL_MAP)
_raw   = df_all["clean_comment"].tolist()
_clean = [_clean(t) for t in _raw]
_labs  = df_all["label"].tolist()

for _s in SEEDS_LR2:
    from sklearn.model_selection import train_test_split as _tts
    _rt, _rtmp, _yt, _ytmp = _tts(_clean, _labs, test_size=0.30,
                                   random_state=_s, stratify=_labs)
    _rv, _rte, _yv, _yte   = _tts(_rtmp, _ytmp, test_size=0.50,
                                   random_state=_s, stratify=_ytmp)
    from sklearn.feature_extraction.text import TfidfVectorizer as _TV
    from sklearn.linear_model import LogisticRegression as _LR
    _vec = _TV(max_features=100000, ngram_range=(1,2), sublinear_tf=True, min_df=2)
    _Xtr = _vec.fit_transform(_rt); _Xte = _vec.transform(_rte)
    _lr  = _LR(max_iter=1000, C=1.0, solver="lbfgs",
               multi_class="multinomial", random_state=_s, n_jobs=-1)
    _lr.fit(_Xtr, _yt)
    _yp = _lr.predict(_Xte)
    from sklearn.metrics import accuracy_score as _ac, f1_score as _fs
    lr2_results.append({"seed":_s,
                         "acc": _ac(_yte,_yp),
                         "f1_w": _fs(_yte,_yp,average="weighted",zero_division=0),
                         "f1_mac": _fs(_yte,_yp,average="macro",zero_division=0)})
    print(f"  Seed {_s:>3}  acc={lr2_results[-1]['acc']:.4f}  "
          f"f1_mac={lr2_results[-1]['f1_mac']:.4f}")

_lr2df = pd.DataFrame(lr2_results).set_index("seed")
print("\n  Part-2 LR — mean ± std across seeds:")
for col in ["acc","f1_w","f1_mac"]:
    print(f"    {col:<8}: {_lr2df[col].mean():.4f} ± {_lr2df[col].std():.4f}")

# ── Train final seed-42 model (used in ensemble below) ───────────────────────
lr_model_obj, tfidf_vec_obj = train_lr(X_train_lr, y_train, X_test_lr, y_test)



── Multi-seed LR variance (Part 2) ─────────────────────────────────
  Seed  42  acc=0.8272  f1_mac=0.8045
  Seed   7  acc=0.8238  f1_mac=0.8010
  Seed  13  acc=0.8283  f1_mac=0.8067
  Seed  21  acc=0.8337  f1_mac=0.8140
  Seed  99  acc=0.8279  f1_mac=0.8074

  Part-2 LR — mean ± std across seeds:
    acc     : 0.8282 ± 0.0036
    f1_w    : 0.8228 ± 0.0041
    f1_mac  : 0.8067 ± 0.0048
[INFO] Fitting TF-IDF vectorizer ...
[INFO] Training Logistic Regression ...
[✓] LR Test Accuracy    : 0.8272
[✓] LR Weighted F1      : 0.8214
[✓] LR Macro F1         : 0.8045
[✓] Saved model      → ./lr_model.joblib
[✓] Saved vectorizer → ./tfidf_vectorizer.joblib


In [ ]:
# ============================================================
# HYBRID ENSEMBLE — PART 2 PIPELINE STARTS HERE
# (DeBERTa uses raw text | LR uses cleaned text)
# ============================================================

In [10]:
# ╔══════════════════════════════════════════════════════════════╗
# ║     SECTION 1 — SETUP & DATA (matched to v2 training)      ║
# ╚══════════════════════════════════════════════════════════════╝

import warnings
warnings.filterwarnings("ignore")
import os, re, string, random, json, time
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import logging
logging.getLogger("transformers").setLevel(logging.ERROR)

import numpy as np
import pandas as pd
from tqdm import tqdm
import torch
import joblib
import nltk
nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    cohen_kappa_score, matthews_corrcoef, roc_auc_score, confusion_matrix
)
from statsmodels.stats.contingency_tables import mcnemar as mcnemar_test
from transformers import AutoTokenizer, AutoModelForSequenceClassification

SEED       = 42
LABEL_MAP  = {-1: 0, 0: 1, 1: 2}
CLASS_NAMES = ["Negative", "Neutral", "Positive"]
DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
STOP_WORDS = set(stopwords.words("english"))

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
print(f"[INFO] Device: {DEVICE}")

# ── Text cleaner used ONLY for LR (LR was trained on cleaned text) ────────
def clean_text(text: str) -> str:
    """Lowercase, strip URLs/punctuation, remove NLTK stopwords."""
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    return " ".join(t for t in text.split() if t not in STOP_WORDS)

# ── Load data — mirrors v2 training script EXACTLY ───────────────────────
df = pd.read_csv("Reddit_Data.csv")
df = df.dropna(subset=["clean_comment", "category"])
df["clean_comment"] = df["clean_comment"].astype(str).str.strip()
df = df[df["clean_comment"].str.len() > 0].reset_index(drop=True)
df["label"] = df["category"].map(LABEL_MAP)
print(f"[INFO] Rows: {len(df)}")

raw_texts    = df["clean_comment"].tolist()   # ← for DeBERTa (no extra cleaning)
labels_all   = df["label"].tolist()
cleaned_texts = [clean_text(t) for t in raw_texts]  # ← for LR only

# ── Identical split to v2 training (test_size=0.30 first, then 50/50) ────
X_tr_r, X_tmp_r, y_tr, y_tmp = train_test_split(
    raw_texts, labels_all, test_size=0.30,
    random_state=SEED, stratify=labels_all)
X_val_r, X_te_r, y_val, y_te = train_test_split(
    X_tmp_r, y_tmp, test_size=0.50,
    random_state=SEED, stratify=y_tmp)

# Cleaned versions for LR — same indices
X_tr_c, X_tmp_c, _, _ = train_test_split(
    cleaned_texts, labels_all, test_size=0.30,
    random_state=SEED, stratify=labels_all)
X_val_c, X_te_c, _, _ = train_test_split(
    X_tmp_c, y_tmp, test_size=0.50,
    random_state=SEED, stratify=y_tmp)

y_train = np.array(y_tr)
y_val   = np.array(y_val)
y_test  = np.array(y_te)

# Raw splits for DeBERTa
X_train_raw = X_tr_r;  X_val_raw = X_val_r;  X_test_raw = X_te_r
# Cleaned splits for LR
X_train_lr  = X_tr_c;  X_val_lr  = X_val_c;  X_test_lr  = X_te_c

print(f"[INFO] train={len(y_train)}  val={len(y_val)}  test={len(y_test)}")
print(f"[INFO] DeBERTa uses raw clean_comment — no extra preprocessing")
print(f"[INFO] LR uses cleaned text — matches LR training preprocessing")


[INFO] Device: cuda
[INFO] Rows: 37028
[INFO] train=25919  val=5554  test=5555
[INFO] DeBERTa uses raw clean_comment — no extra preprocessing
[INFO] LR uses cleaned text — matches LR training preprocessing


In [11]:
# ╔══════════════════════════════════════════════════════════════╗
# ║               SECTION 2 — LOAD SAVED MODELS                ║
# ╚══════════════════════════════════════════════════════════════╝

print("="*60)
print("  SECTION 2 — LOADING SAVED MODELS")
print("="*60)

def load_lr_model(model_path="./lr_model.joblib", vec_path="./tfidf_vectorizer.joblib"):
    """Load pre-trained LR and TF-IDF vectorizer."""
    return joblib.load(model_path), joblib.load(vec_path)

def load_deberta_model(model_dir="./deberta_sentiment_v2"):
    """Load DeBERTa from HuggingFace saved format, force FP32."""
    tok = AutoTokenizer.from_pretrained(model_dir)
    mdl = AutoModelForSequenceClassification.from_pretrained(model_dir)
    mdl.to(DEVICE).float().eval()
    return tok, mdl

lr_model, tfidf_vec = load_lr_model()
print("[✓] LR + TF-IDF loaded")

deberta_tokenizer, deberta_model = load_deberta_model()
print(f"[✓] DeBERTa loaded on {DEVICE} (FP32)")

# ── Inference functions ───────────────────────────────────────────────────
def lr_predict_proba(texts, vectorizer, model):
    """TF-IDF → LR softmax probabilities [N, 3]. Uses cleaned text."""
    return model.predict_proba(vectorizer.transform(texts))

def deberta_predict_proba(texts, tokenizer, model, batch_size=64):
    """Batched DeBERTa inference. Uses RAW clean_comment — no extra cleaning."""
    all_probs = []
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc="DeBERTa", leave=False):
            batch = texts[i:i+batch_size]
            enc   = tokenizer(batch, truncation=True, padding=True,
                              max_length=128, return_tensors="pt")
            enc   = {k: v.to(DEVICE) for k, v in enc.items()}
            logits = model(**enc).logits
            probs  = torch.softmax(logits.float(), dim=-1).cpu().numpy()
            all_probs.append(probs)
    return np.vstack(all_probs)

# ── Test-set inference ────────────────────────────────────────────────────
print("\n[INFO] Running test-set inference ...")

t0 = time.perf_counter()
lr_probs_test  = lr_predict_proba(X_test_lr, tfidf_vec, lr_model)
lr_time_total  = time.perf_counter() - t0
lr_preds_test  = np.argmax(lr_probs_test, axis=1)

if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
t0 = time.perf_counter()
deb_probs_test = deberta_predict_proba(X_test_raw, deberta_tokenizer, deberta_model)
deb_time_total = time.perf_counter() - t0
deb_peak_mem   = torch.cuda.max_memory_allocated() / 1e6 if torch.cuda.is_available() else 0.0
deb_preds_test = np.argmax(deb_probs_test, axis=1)

n_test     = len(y_test)
lr_ms_per  = lr_time_total  * 1000 / n_test
deb_ms_per = deb_time_total * 1000 / n_test

lr_acc  = accuracy_score(y_test, lr_preds_test)
deb_acc = accuracy_score(y_test, deb_preds_test)

print("\n┌──────────────────────────────────────────────────────┐")
print("│          MODEL LOADING CONFIRMATION TABLE            │")
print("├──────────────────────────┬───────────┬──────────────┤")
print("│ Model                    │ Test Acc  │ Input Type   │")
print("├──────────────────────────┼───────────┼──────────────┤")
print(f"│ Logistic Regression      │  {lr_acc:.4f}  │ cleaned text │")
print(f"│ DeBERTa-v3-small         │  {deb_acc:.4f}  │ raw text     │")
print("└──────────────────────────┴───────────┴──────────────┘")


  SECTION 2 — LOADING SAVED MODELS
[✓] LR + TF-IDF loaded


Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

[✓] DeBERTa loaded on cuda (FP32)

[INFO] Running test-set inference ...



┌──────────────────────────────────────────────────────┐
│          MODEL LOADING CONFIRMATION TABLE            │
├──────────────────────────┬───────────┬──────────────┤
│ Model                    │ Test Acc  │ Input Type   │
├──────────────────────────┼───────────┼──────────────┤
│ Logistic Regression      │  0.8272  │ cleaned text │
│ DeBERTa-v3-small         │  0.9401  │ raw text     │
└──────────────────────────┴───────────┴──────────────┘


In [12]:
# ╔══════════════════════════════════════════════════════════════╗
# ║               SECTION 3 — ALPHA TUNING (VAL SET)           ║
# ╚══════════════════════════════════════════════════════════════╝

print("\n" + "="*60)
print("  SECTION 3 — ALPHA GRID SEARCH (VALIDATION SET)")
print("="*60)

ALPHA_GRID = [0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90]

print("[INFO] Val-set inference ...")
lr_probs_val  = lr_predict_proba(X_val_lr,  tfidf_vec, lr_model)
deb_probs_val = deberta_predict_proba(X_val_raw, deberta_tokenizer, deberta_model)

print("\n┌───────┬──────────────┬──────────────────┬──────────────┐")
print("│ Alpha │ Val Accuracy │ Val Weighted-F1  │ Val Macro-F1 │")
print("├───────┼──────────────┼──────────────────┼──────────────┤")

best_alpha    = ALPHA_GRID[0]
best_macro_f1 = -1.0
alpha_results = []

for alpha in ALPHA_GRID:
    p_ens  = alpha * deb_probs_val + (1 - alpha) * lr_probs_val
    preds  = np.argmax(p_ens, axis=1)
    acc    = accuracy_score(y_val, preds)
    wt_f1  = f1_score(y_val, preds, average="weighted")
    mac_f1 = f1_score(y_val, preds, average="macro")
    alpha_results.append((alpha, acc, wt_f1, mac_f1))
    marker = " ◄" if mac_f1 > best_macro_f1 else ""
    if mac_f1 > best_macro_f1:
        best_macro_f1 = mac_f1
        best_alpha    = alpha
    print(f"│ {alpha:.2f}  │   {acc:.4f}   │     {wt_f1:.4f}       │   {mac_f1:.4f}   │{marker}")

print("└───────┴──────────────┴──────────────────┴──────────────┘")
print(f"\n[✓] Best alpha = {best_alpha}  (val macro-F1 = {best_macro_f1:.4f})")



  SECTION 3 — ALPHA GRID SEARCH (VALIDATION SET)
[INFO] Val-set inference ...



┌───────┬──────────────┬──────────────────┬──────────────┐
│ Alpha │ Val Accuracy │ Val Weighted-F1  │ Val Macro-F1 │
├───────┼──────────────┼──────────────────┼──────────────┤
│ 0.60  │   0.9467   │     0.9466       │   0.9415   │ ◄
│ 0.65  │   0.9476   │     0.9476       │   0.9425   │ ◄
│ 0.70  │   0.9478   │     0.9478       │   0.9428   │ ◄
│ 0.75  │   0.9481   │     0.9482       │   0.9432   │ ◄
│ 0.80  │   0.9478   │     0.9478       │   0.9429   │
│ 0.85  │   0.9480   │     0.9480       │   0.9431   │
│ 0.90  │   0.9474   │     0.9475       │   0.9424   │
└───────┴──────────────┴──────────────────┴──────────────┘

[✓] Best alpha = 0.75  (val macro-F1 = 0.9432)


In [14]:
# ╔══════════════════════════════════════════════════════════════╗
# ║         SECTION 5 — IEEE-GRADE METRICS REPORT               ║
# ╚══════════════════════════════════════════════════════════════╝

print("\n" + "="*60)
print("  SECTION 5 — IEEE-GRADE METRICS REPORT")
print("="*60)


def compute_all_metrics(y_true: np.ndarray, y_pred: np.ndarray,
                        y_proba: np.ndarray, model_name: str) -> dict:
    """
    Compute the full suite of classification metrics for one model.

    Parameters
    ----------
    y_true     : ground-truth labels
    y_pred     : hard predictions
    y_proba    : softmax probabilities [N, 3]
    model_name : string label for printing

    Returns
    -------
    metrics : dict
    """
    acc    = accuracy_score(y_true, y_pred)
    mac_p  = precision_score(y_true, y_pred, average="macro",    zero_division=0)
    mac_r  = recall_score   (y_true, y_pred, average="macro",    zero_division=0)
    mac_f1 = f1_score       (y_true, y_pred, average="macro",    zero_division=0)
    wt_p   = precision_score(y_true, y_pred, average="weighted", zero_division=0)
    wt_r   = recall_score   (y_true, y_pred, average="weighted", zero_division=0)
    wt_f1  = f1_score       (y_true, y_pred, average="weighted", zero_division=0)
    kappa  = cohen_kappa_score(y_true, y_pred)
    mcc    = matthews_corrcoef(y_true, y_pred)
    roc    = roc_auc_score(y_true, y_proba, multi_class="ovr", average="macro")

    # Per-class metrics
    per_class_p = precision_score(y_true, y_pred, average=None, zero_division=0)
    per_class_r = recall_score   (y_true, y_pred, average=None, zero_division=0)
    per_class_f = f1_score       (y_true, y_pred, average=None, zero_division=0)
    supports    = [np.sum(y_true == c) for c in range(3)]

    return {
        "name":     model_name,
        "acc":      acc,
        "mac_p":    mac_p,
        "mac_r":    mac_r,
        "mac_f1":   mac_f1,
        "wt_p":     wt_p,
        "wt_r":     wt_r,
        "wt_f1":    wt_f1,
        "kappa":    kappa,
        "mcc":      mcc,
        "roc_auc":  roc,
        "per_class_p": per_class_p.tolist(),
        "per_class_r": per_class_r.tolist(),
        "per_class_f": per_class_f.tolist(),
        "supports":    supports,
    }


metrics_lr  = compute_all_metrics(y_test, lr_preds_test,  lr_probs_test,  "Logistic Regression")
metrics_deb = compute_all_metrics(y_test, deb_preds_test, deb_probs_test, "DeBERTa-v3-small")
# ── Compute ensemble predictions on test set ─────────────────────────────────
print("Computing ensemble predictions on test set...")

# DeBERTa predictions
deb_probs_test = []
with torch.no_grad():
    for batch in test_loader:
        ids  = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            logits = deberta_model(input_ids=ids, attention_mask=mask).logits
        probs = torch.softmax(logits.float(), dim=1).cpu().numpy()
        deb_probs_test.extend(probs)
deb_probs_test = np.array(deb_probs_test)
deb_preds_test = deb_probs_test.argmax(axis=1)

# LR predictions
lr_vec_test = tfidf_vec_obj.transform(X_test_lr)
lr_probs_test = lr_model_obj.predict_proba(lr_vec_test)
lr_preds_test = lr_probs_test.argmax(axis=1)

# Ensemble (weighted average)
ens_probs_test = best_alpha * deb_probs_test + (1 - best_alpha) * lr_probs_test
ens_preds_test = ens_probs_test.argmax(axis=1)

print(f"  DeBERTa test acc: {accuracy_score(y_test, deb_preds_test):.4f}")
print(f"  LR test acc:      {accuracy_score(y_test, lr_preds_test):.4f}")
print(f"  Ensemble acc:     {accuracy_score(y_test, ens_preds_test):.4f}\n")

# ──────────────────────────────────────────────────────────────────────────────
metrics_ens = compute_all_metrics(y_test, ens_preds_test, ens_probs_test, "Hybrid Ensemble")

ALL_METRICS = [metrics_lr, metrics_deb, metrics_ens]


def print_overall_metrics(m: dict):
    """Print formatted overall metrics for one model."""
    print(f"\n  ── {m['name']} ──")
    print(f"  Accuracy          : {m['acc']:.4f}")
    print(f"  Macro  P/R/F1     : {m['mac_p']:.4f} / {m['mac_r']:.4f} / {m['mac_f1']:.4f}")
    print(f"  Weighted P/R/F1   : {m['wt_p']:.4f} / {m['wt_r']:.4f} / {m['wt_f1']:.4f}")
    print(f"  Cohen's Kappa     : {m['kappa']:.4f}")
    print(f"  MCC               : {m['mcc']:.4f}")
    print(f"  ROC-AUC (macro)   : {m['roc_auc']:.4f}")


# ── A. Overall Metrics ────────────────────────────────────────
print("\n[A] OVERALL METRICS")
for m in ALL_METRICS:
    print_overall_metrics(m)


# ── B. Per-class Metrics ──────────────────────────────────────
print("\n[B] PER-CLASS METRICS")
for m in ALL_METRICS:
    print(f"\n  {m['name']}")
    print(f"  {'Class':<12} {'Precision':>10} {'Recall':>10} {'F1-Score':>10} {'Support':>9}")
    print(f"  {'-'*55}")
    for i, cls in enumerate(CLASS_NAMES):
        print(f"  {cls:<12} {m['per_class_p'][i]:>10.4f} {m['per_class_r'][i]:>10.4f} "
              f"{m['per_class_f'][i]:>10.4f} {m['supports'][i]:>9}")


# ── C. ROC-AUC ───────────────────────────────────────────────
print("\n[C] ROC-AUC (one-vs-rest, macro-averaged)")
for m in ALL_METRICS:
    print(f"  {m['name']:<25}: {m['roc_auc']:.4f}")


# ── D. McNemar's Test: Ensemble vs DeBERTa ───────────────────
print("\n[D] STATISTICAL SIGNIFICANCE — McNemar's Test (Ensemble vs DeBERTa)")

ens_correct = (ens_preds_test == y_test)
deb_correct = (deb_preds_test == y_test)

# Contingency table
b = np.sum( ens_correct & ~deb_correct)  # Ens correct, DeBERTa wrong
c = np.sum(~ens_correct &  deb_correct)  # Ens wrong,   DeBERTa correct

contingency_table = np.array([[np.sum( ens_correct &  deb_correct),  b],
                               [c,  np.sum(~ens_correct & ~deb_correct)]])

result  = mcnemar_test(contingency_table, exact=False, correction=True)
chi2    = result.statistic
p_value = result.pvalue

verdict = ("statistically significant improvement" if p_value < 0.05
           else "no statistically significant difference")

print(f"  Contingency (b={b}, c={c})")
print(f"  Chi² statistic : {chi2:.4f}")
print(f"  p-value        : {p_value:.6f}")
print(f"  Verdict        : {verdict} (α=0.05)")


# ── E. Computational Cost ─────────────────────────────────────
print("\n[E] COMPUTATIONAL COST COMPARISON")

# Count DeBERTa parameters
n_params_deb = sum(p.numel() for p in deberta_model.parameters()) / 1e6

print(f"\n  {'Model':<25} {'Params (M)':>12} {'ms/sample':>12} {'Peak GPU (MB)':>14}")
print(f"  {'-'*67}")
print(f"  {'Logistic Regression':<25} {'~0.001':>12} {lr_ms_per:>12.3f} {'N/A':>14}")
print(f"  {'DeBERTa-v3-small':<25} {n_params_deb:>12.1f} {deb_ms_per:>12.3f} {deb_peak_mem:>14.1f}")
print(f"  {'Hybrid Ensemble':<25} {n_params_deb:>12.1f} {(lr_ms_per+deb_ms_per):>12.3f} {deb_peak_mem:>14.1f}")


  SECTION 5 — IEEE-GRADE METRICS REPORT
Computing ensemble predictions on test set...
  DeBERTa test acc: 0.9402
  LR test acc:      0.8272
  Ensemble acc:     0.9422


[A] OVERALL METRICS

  ── Logistic Regression ──
  Accuracy          : 0.8272
  Macro  P/R/F1     : 0.8358 / 0.7940 / 0.8045
  Weighted P/R/F1   : 0.8306 / 0.8272 / 0.8214
  Cohen's Kappa     : 0.7273
  MCC               : 0.7321
  ROC-AUC (macro)   : 0.9422

  ── DeBERTa-v3-small ──
  Accuracy          : 0.9401
  Macro  P/R/F1     : 0.9327 / 0.9369 / 0.9346
  Weighted P/R/F1   : 0.9410 / 0.9401 / 0.9404
  Cohen's Kappa     : 0.9073
  MCC               : 0.9075
  ROC-AUC (macro)   : 0.9880

  ── Hybrid Ensemble ──
  Accuracy          : 0.9422
  Macro  P/R/F1     : 0.9358 / 0.9378 / 0.9367
  Weighted P/R/F1   : 0.9428 / 0.9422 / 0.9424
  Cohen's Kappa     : 0.9106
  MCC               : 0.9106
  ROC-AUC (macro)   : 0.9890

[B] PER-CLASS METRICS

  Logistic Regression
  Class         Precision     Recall   F1-Score   Supp

In [15]:
# ╔══════════════════════════════════════════════════════════════╗
# ║               SECTION 6 — CONFUSION MATRICES                ║
# ╚══════════════════════════════════════════════════════════════╝

print("\n" + "="*60)
print("  SECTION 6 — CONFUSION MATRICES")
print("="*60)


def plot_confusion_matrices(models_info: list, save_path: str = "confusion_matrices_comparison.png"):
    """
    Plot side-by-side 3x3 confusion matrices for all models.

    Parameters
    ----------
    models_info : list of (name, y_true, y_pred) tuples
    save_path   : output file path
    """
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle("Confusion Matrices — Sentiment Classification", fontsize=14, fontweight="bold")

    for ax, (name, y_true, y_pred) in zip(axes, models_info):
        cm    = confusion_matrix(y_true, y_pred)
        cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

        # Annotations: count + percentage
        annot = np.empty_like(cm, dtype=object)
        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                annot[i, j] = f"{cm[i,j]}\n({cm_pct[i,j]:.1f}%)"

        sns.heatmap(cm, annot=annot, fmt="", cmap="Blues", ax=ax,
                    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                    linewidths=0.5, cbar_kws={"shrink": 0.8})
        ax.set_title(name, fontsize=12, fontweight="bold")
        ax.set_xlabel("Predicted Label", fontsize=10)
        ax.set_ylabel("True Label", fontsize=10)
        ax.tick_params(axis="x", rotation=30)
        ax.tick_params(axis="y", rotation=0)

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"[✓] Confusion matrices saved to {save_path}")


cm_models = [
    ("Logistic Regression", y_test, lr_preds_test),
    ("DeBERTa-v3-small",    y_test, deb_preds_test),
    ("Hybrid Ensemble",     y_test, ens_preds_test),
]
plot_confusion_matrices(cm_models)


  SECTION 6 — CONFUSION MATRICES
[✓] Confusion matrices saved to confusion_matrices_comparison.png


In [16]:
# ╔══════════════════════════════════════════════════════════════╗
# ║               SECTION 7 — ROC CURVES                        ║
# ╚══════════════════════════════════════════════════════════════╝

print("\n" + "="*60)
print("  SECTION 7 — ROC CURVES")
print("="*60)

from sklearn.metrics import roc_curve
from sklearn.preprocessing import label_binarize


def plot_roc_curves(models_roc: list, save_path: str = "roc_curves_comparison.png"):
    """
    Plot one-vs-rest ROC curves for all 3 classes for each model.

    Parameters
    ----------
    models_roc : list of (name, y_true, y_proba) tuples
    save_path  : output file path
    """
    colors = ["#e41a1c", "#377eb8", "#4daf4a"]
    y_bin  = label_binarize(y_test, classes=[0, 1, 2])

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle("One-vs-Rest ROC Curves — Sentiment Classification",
                 fontsize=14, fontweight="bold")

    for ax, (name, y_true, y_proba) in zip(axes, models_roc):
        y_b = label_binarize(y_true, classes=[0, 1, 2])
        for i, (cls, col) in enumerate(zip(CLASS_NAMES, colors)):
            fpr, tpr, _ = roc_curve(y_b[:, i], y_proba[:, i])
            auc_val     = roc_auc_score(y_b[:, i], y_proba[:, i])
            ax.plot(fpr, tpr, color=col, lw=2,
                    label=f"{cls} (AUC={auc_val:.3f})")
        ax.plot([0, 1], [0, 1], "k--", lw=1, alpha=0.6)
        ax.set_xlim([0, 1]); ax.set_ylim([0, 1.02])
        ax.set_xlabel("False Positive Rate", fontsize=10)
        ax.set_ylabel("True Positive Rate",  fontsize=10)
        ax.set_title(name, fontsize=12, fontweight="bold")
        ax.legend(loc="lower right", fontsize=9)
        ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"[✓] ROC curves saved to {save_path}")


roc_models = [
    ("Logistic Regression", y_test, lr_probs_test),
    ("DeBERTa-v3-small",    y_test, deb_probs_test),
    ("Hybrid Ensemble",     y_test, ens_probs_test),
]
plot_roc_curves(roc_models)


  SECTION 7 — ROC CURVES
[✓] ROC curves saved to roc_curves_comparison.png


In [17]:
# ╔══════════════════════════════════════════════════════════════╗
# ║         SECTION 8 — MASTER SUMMARY TABLE (IEEE-READY)       ║
# ╚══════════════════════════════════════════════════════════════╝

print("\n" + "="*60)
print("  SECTION 8 — IEEE-READY MASTER COMPARISON TABLE")
print("="*60)

COLS    = ["Model", "Acc", "Mac-P", "Mac-R", "Mac-F1", "Wt-F1", "Kappa", "MCC", "ROC-AUC"]
COL_W   = [26, 8, 8, 8, 8, 8, 8, 8, 9]
ROW_SEP = "+" + "+".join("-"*w for w in COL_W) + "+"

def fmt_row(values):
    """Format a table row with fixed-width columns."""
    parts = []
    for v, w in zip(values, COL_W):
        parts.append(str(v).center(w))
    return "|" + "|".join(parts) + "|"

print(ROW_SEP)
print(fmt_row(COLS))
print(ROW_SEP)

rows_data = []
for m in ALL_METRICS:
    rows_data.append([
        m["name"], m["acc"], m["mac_p"], m["mac_r"],
        m["mac_f1"], m["wt_f1"], m["kappa"], m["mcc"], m["roc_auc"]
    ])
    print(fmt_row([m["name"]] + [f"{v:.4f}" for v in [
        m["acc"], m["mac_p"], m["mac_r"], m["mac_f1"],
        m["wt_f1"], m["kappa"], m["mcc"], m["roc_auc"]
    ]]))

print(ROW_SEP)

# ── BEST markers ──────────────────────────────────────────────
numeric_cols = list(range(1, 9))
best_vals, best_row = [], []

for col_idx in numeric_cols:
    col_vals = [rows_data[r][col_idx] for r in range(len(rows_data))]
    best_v   = max(col_vals)
    best_r   = col_vals.index(best_v)
    best_vals.append(best_v)
    best_row.append(f"{best_v:.4f}({'LR' if best_r==0 else 'DB' if best_r==1 else 'EN'})")

print(fmt_row(["BEST ↑"] + best_row))
print(ROW_SEP)
print("\n  LR=Logistic Regression, DB=DeBERTa-v3-small, EN=Hybrid Ensemble")


  SECTION 8 — IEEE-READY MASTER COMPARISON TABLE
+--------------------------+--------+--------+--------+--------+--------+--------+--------+---------+
|          Model           |  Acc   | Mac-P  | Mac-R  | Mac-F1 | Wt-F1  | Kappa  |  MCC   | ROC-AUC |
+--------------------------+--------+--------+--------+--------+--------+--------+--------+---------+
|   Logistic Regression    | 0.8272 | 0.8358 | 0.7940 | 0.8045 | 0.8214 | 0.7273 | 0.7321 |  0.9422 |
|     DeBERTa-v3-small     | 0.9401 | 0.9327 | 0.9369 | 0.9346 | 0.9404 | 0.9073 | 0.9075 |  0.9880 |
|     Hybrid Ensemble      | 0.9422 | 0.9358 | 0.9378 | 0.9367 | 0.9424 | 0.9106 | 0.9106 |  0.9890 |
+--------------------------+--------+--------+--------+--------+--------+--------+--------+---------+
|          BEST ↑          |0.9422(EN)|0.9358(EN)|0.9378(EN)|0.9367(EN)|0.9424(EN)|0.9106(EN)|0.9106(EN)|0.9890(EN)|
+--------------------------+--------+--------+--------+--------+--------+--------+--------+---------+

  LR=Logistic Re

In [20]:
# ╔══════════════════════════════════════════════════════════════╗
# ║               SECTION 9 — SAVE RESULTS                      ║
# ╚══════════════════════════════════════════════════════════════╝

print("\n" + "="*60)
print("  SECTION 9 — SAVING RESULTS")
print("="*60)


def save_results(metrics_list: list, json_path: str = "results_hybrid_ensemble.json"):
    """
    Serialise full metrics dictionary to JSON.

    Parameters
    ----------
    metrics_list : list of metric dicts
    json_path    : output file path
    """
    payload = {
        "best_alpha":       best_alpha,
        "alpha_grid_search": [
            {"alpha": a, "val_acc": acc, "val_wt_f1": wf, "val_mac_f1": mf}
            for a, acc, wf, mf in alpha_results
        ],
        "mcnemar": {
            "chi2": float(chi2),
            "p_value": float(p_value),
            "verdict": verdict,
        },
        "computational_cost": {
            "LR":  {"params_M": 0.001, "ms_per_sample": lr_ms_per,  "peak_gpu_mb": 0.0},
            "DeBERTa": {"params_M": n_params_deb, "ms_per_sample": deb_ms_per, "peak_gpu_mb": deb_peak_mem},
        },
        "models": {m["name"]: {k: v for k, v in m.items() if k != "name"}
                   for m in metrics_list},
    }
    with open(json_path, "w") as f:
        json.dump(payload, f, indent=2, default=lambda o: int(o) if isinstance(o, np.integer) else float(o) if isinstance(o, np.floating) else str(o))
    print(f"[✓] Metrics saved to {json_path}")


def save_predictions(csv_path: str = "predictions_comparison.csv"):
    """
    Save all model predictions and ground truth to CSV.

    Parameters
    ----------
    csv_path : output file path
    """
    df_out = pd.DataFrame({
        "ground_truth":    y_test,
        "lr_pred":         lr_preds_test,
        "deberta_pred":    deb_preds_test,
        "ensemble_pred":   ens_preds_test,
        "lr_prob_neg":     lr_probs_test[:, 0],
        "lr_prob_neu":     lr_probs_test[:, 1],
        "lr_prob_pos":     lr_probs_test[:, 2],
        "deb_prob_neg":    deb_probs_test[:, 0],
        "deb_prob_neu":    deb_probs_test[:, 1],
        "deb_prob_pos":    deb_probs_test[:, 2],
        "ens_prob_neg":    ens_probs_test[:, 0],
        "ens_prob_neu":    ens_probs_test[:, 1],
        "ens_prob_pos":    ens_probs_test[:, 2],
    })
    df_out.to_csv(csv_path, index=False)
    print(f"[✓] Predictions saved to {csv_path}")


save_results(ALL_METRICS)
save_predictions()

print("\n" + "="*60)
print("  ALL SECTIONS COMPLETE ✓")
print(f"  Best alpha           : {best_alpha}")
print(f"  Ensemble Test Acc    : {metrics_ens['acc']:.4f}")
print(f"  Ensemble Macro-F1    : {metrics_ens['mac_f1']:.4f}")
print(f"  Ensemble Weighted-F1 : {metrics_ens['wt_f1']:.4f}")
print(f"  Ensemble ROC-AUC     : {metrics_ens['roc_auc']:.4f}")
print("="*60)


  SECTION 9 — SAVING RESULTS
[✓] Metrics saved to results_hybrid_ensemble.json
[✓] Predictions saved to predictions_comparison.csv

  ALL SECTIONS COMPLETE ✓
  Best alpha           : 0.75
  Ensemble Test Acc    : 0.9422
  Ensemble Macro-F1    : 0.9367
  Ensemble Weighted-F1 : 0.9424
  Ensemble ROC-AUC     : 0.9890
